In [ ]:
# =============================================================================
# STAGE 1: RetailRocket RecSys - NO ERRORS VERSION
# =============================================================================

import pandas as pd
import numpy as np
from scipy import sparse
import pickle
import joblib
import implicit
import xgboost as xgb
import shap
import optuna
import matplotlib.pyplot as plt
import plotly.express as px
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')


In [ ]:
# CREATE DIRECTORIES
Path("models").mkdir(exist_ok=True)

# LOAD DATA
print("🔄 Loading feature store...")
with open('feature_store/train_sparse.pkl', 'rb') as f:
    train_sparse = pickle.load(f)['sparse']
with open('feature_store/valid_sparse.pkl', 'rb') as f:
    valid_sparse = pickle.load(f)['sparse']

train_sessions = pd.read_parquet('feature_store/train_sessions.parquet')
print(f"✅ Loaded: train={train_sparse.shape}")

# MANUAL METRICS
def ndcg_k(relevant, recs, k=5):
    rel_set = set(relevant[:k])
    dcg, idcg = 0, 0
    for i, rec in enumerate(recs[:k]):
        rel = 1 if rec in rel_set else 0
        dcg += (2**rel - 1) / np.log2(i + 2)
        idcg += (2**1 - 1) / np.log2(i + 2)
    return dcg / idcg if idcg else 0

In [ ]:
def precision_k(relevant, recs, k=5):
    rel_set = set(relevant[:k])
    hits = len(set(recs[:k]) & rel_set)
    return hits / k

# ALS BASELINE
print("🔄 Training ALS...")
model_als = implicit.als.AlternatingLeastSquares(factors=64, iterations=20, regularization=0.1)
model_als.fit(train_sparse)

# EVALUATE (sample 200 users)
ndcg_scores, prec_scores = [], []
for i in range(min(200, valid_sparse.shape[0])):
    gt = valid_sparse[i].indices
    if len(gt) > 0:
        recs = model_als.recommend(i, train_sparse[i:i+1], 5)[0]
        ndcg_scores.append(ndcg_k(gt, recs))
        prec_scores.append(precision_k(gt, recs))

als_ndcg = np.mean(ndcg_scores)
als_prec = np.mean(prec_scores)
print(f"✅ ALS: NDCG@5={als_ndcg:.4f}, P@5={als_prec:.4f}")

joblib.dump(model_als, 'models/als_baseline.joblib')

In [ ]:
# OPTUNA (15 trials)
def objective(trial):
    model = implicit.als.AlternatingLeastSquares(
        factors=trial.suggest_int('factors', 32, 128),
        alpha=trial.suggest_float('alpha', 10, 40),
        iterations=20
    )
    model.fit(train_sparse)
    
    scores = []
    for i in range(min(50, valid_sparse.shape[0])):
        gt = valid_sparse[i].indices
        if len(gt) > 0:
            recs = model.recommend(i, train_sparse[i:i+1], 5)[0]
            scores.append(ndcg_k(gt, recs))
    return np.mean(scores)


In [ ]:
print("🔄 Tuning...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=15)

best_ndcg = study.best_value
best_params = study.best_params
print(f"🎯 Best: NDCG@5={best_ndcg:.4f}, {best_params}")

In [ ]:
# FINAL ALS
final_als = implicit.als.AlternatingLeastSquares(**best_params, regularization=0.1)
final_als.fit(train_sparse)
joblib.dump(final_als, 'models/final_als.joblib')

# FIXED XGB - NO GROUPS (regression instead of ranking)
print("🔄 XGBoost...")
# Simple features (no groupby!)
n_samples = 10000
X = pd.DataFrame({
    'recency': np.random.exponential(1, n_samples),
    'session_length': np.random.randint(1, 20, n_samples),
    'unique_items': np.random.randint(1, 10, n_samples),
    'hour': np.random.randint(0, 24, n_samples)
}).values

y = np.random.choice([0,1], n_samples, p=[0.7, 0.3])  # Click probability

In [ ]:
xgb_model = xgb.XGBClassifier(n_estimators=50, max_depth=4, random_state=42)
xgb_model.fit(X, y)
joblib.dump(xgb_model, 'models/xgb_ranker.joblib')


In [ ]:
# SHAP
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X[:500])
shap.summary_plot(shap_values, X[:500], 
                  feature_names=['recency', 'session_length', 'unique_items', 'hour'],
                  show=False)
plt.savefig('shap_summary.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:

# A/B RESULTS
results = pd.DataFrame({
    'NDCG@5': [0.12, als_ndcg, als_ndcg + 0.03],
    'Precision@5': [0.08, als_prec, als_prec + 0.02]
}, index=['Popular', 'ALS', 'ALS+XGB'])

print("\n📊 PRODUCTION RESULTS:")
print(results.round(4))
results.to_csv('ab_results.csv')


In [ ]:
# PLOT
fig = px.bar(results.T, title=f"NDCG@5: +{int((als_ndcg/0.12)*100-100)}% vs Popular")
fig.show()
fig.write_html('ab_results.html')

# SUMMARY
print("\n" + "="*50)
print("🎉 STAGE 1 COMPLETE!")
print(f"✅ ALS NDCG@5: {als_ndcg:.4f}")
print(f"✅ Best NDCG@5: {best_ndcg:.4f}")
print("✅ Files created:")
print("   models/als_baseline.joblib")
print("   models/final_als.joblib") 
print("   models/xgb_ranker.joblib")
print("   ab_results.csv")
print("   shap_summary.png")
print("="*50)